### **Load Raw Datasets**

In [1]:
import pandas as pd
import numpy as np
import os

# Define paths
RAW_DIR = "../data/raw"

# Load datasets
df_ember = pd.read_csv(os.path.join(RAW_DIR, "ember_yearly_generation.csv"), low_memory=False)
df_wb = pd.read_csv(os.path.join(RAW_DIR, "world_bank_electricity_access.csv"), low_memory=False)
df_irena = pd.read_csv(os.path.join(RAW_DIR, "irenastat_capacity.csv"), low_memory=False)

print("Datasets successfully loaded into notebook for cleaning tests!")
print(f"Ember Shape: {df_ember.shape}")
print(f"World Bank Shape: {df_wb.shape}")
print(f"IRENA Shape: {df_irena.shape}")

Datasets successfully loaded into notebook for cleaning tests!
Ember Shape: (104203, 24)
World Bank Shape: (265, 39)
IRENA Shape: (288, 5)


#### **Clean & Filter Ember Data**

In [2]:
# Filter Ember for African continent records
df_ember_africa = df_ember[df_ember['Continent'] == 'Africa'].copy()

# Drop rows missing ISO 3 codes (which typically correspond to loose regional groupings)
df_ember_clean = df_ember_africa.dropna(subset=['ISO 3 code']).copy()

# Check unique countries remaining
print(f"Total African countries/territories in clean Ember dataset: {df_ember_clean['Area'].nunique()}")
print("Sample countries:", df_ember_clean['Area'].unique()[:10])

# Quick check for negative generation values (a common dirty data check)
negative_gen = df_ember_clean[df_ember_clean['Generation (TWh)'] < 0]
print(f"Number of rows with negative generation: {len(negative_gen)}")

Total African countries/territories in clean Ember dataset: 55
Sample countries: ['Algeria' 'Angola' 'Benin' 'Botswana' 'Burkina Faso' 'Burundi'
 'Cabo Verde' 'Cameroon' 'Central African Republic' 'Chad']
Number of rows with negative generation: 297


#### **Clean IRENA Data & Melt World Bank Data**

In [3]:
# 1. CLEAN IRENA DATA
# Remove any non-numeric strings or commas from statistics and convert to float
df_irena['Electricity generation statistics'] = (
    df_irena['Electricity generation statistics']
    .astype(str)
    .str.replace(',', '', regex=False)
)
df_irena['Value'] = pd.to_numeric(df_irena['Electricity generation statistics'], errors='coerce')

print(f"IRENA Cleaned Shape: {df_irena.shape}")
print(df_irena[['Region', 'Technology', 'Year', 'Value']].head(3))


# 2. MELT WORLD BANK DATA 
# Identify identifier columns vs year columns
id_vars = ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code']
year_vars = [col for col in df_wb.columns if col not in id_vars]

# Melt from wide to long format
df_wb_long = pd.melt(
    df_wb, 
    id_vars=id_vars, 
    value_vars=year_vars, 
    var_name='Year', 
    value_name='Access_to_Electricity_Pct'
)

# Clean Year and Value columns
df_wb_long['Year'] = pd.to_numeric(df_wb_long['Year'], errors='coerce')
df_wb_long['Access_to_Electricity_Pct'] = pd.to_numeric(df_wb_long['Access_to_Electricity_Pct'], errors='coerce')

# Filter World Bank data for valid years (e.g., from 2000 onwards to match Ember)
df_wb_clean = df_wb_long.dropna(subset=['Access_to_Electricity_Pct']).copy()

print(f"\nWorld Bank Melted & Cleaned Shape: {df_wb_clean.shape}")
df_wb_clean.head(3)

IRENA Cleaned Shape: (288, 6)
   Region       Technology  Year     Value
0  Africa  Total renewable  2000  78948.79
1  Africa  Total renewable  2001  85987.30
2  Africa  Total renewable  2002  84118.80

World Bank Melted & Cleaned Shape: (8097, 6)


,Country Name,Country Code,Indicator Name,Indicator Code,Year,Access_to_Electricity_Pct
0,Aruba,ABW,Access to electricity (% of population),EG.ELC.ACCS.ZS,1990,100.0
5,Albania,ALB,Access to electricity (% of population),EG.ELC.ACCS.ZS,1990,100.0
6,Andorra,AND,Access to electricity (% of population),EG.ELC.ACCS.ZS,1990,100.0


#### **Handle Anomalies (Negative Generation) & Save Processed Files**

In [4]:
# 1. HANDLE NEGATIVE GENERATION ANOMALIES IN EMBER 
print(f"Total rows before handling negative generation: {len(df_ember_clean)}")

# Flag or filter out negative generation values (or isolate them for the quality log)
df_ember_clean['Has_Negative_Generation'] = df_ember_clean['Generation (TWh)'] < 0

# For our core analysis dataset, let's filter out or clip negative values, 
# while keeping a record of them for our quality_checks.csv report
df_ember_valid = df_ember_clean[df_ember_clean['Generation (TWh)'] >= 0].copy()

print(f"Total rows after removing negative generation anomalies: {len(df_ember_valid)}")
print(f"Anomalies isolated: {df_ember_clean['Has_Negative_Generation'].sum()} rows flagged.")


# 2. SAVE PROCESSED FILES
# Ensure processed directory exists
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Save cleaned datasets to parquet/csv
df_ember_valid.to_csv(os.path.join(PROCESSED_DIR, "standardized_ember.csv"), index=False)
df_wb_clean.to_csv(os.path.join(PROCESSED_DIR, "standardized_world_bank.csv"), index=False)
df_irena.to_csv(os.path.join(PROCESSED_DIR, "standardized_irena.csv"), index=False)

print("\nAll cleaned and standardized datasets successfully saved to data/processed/!")

Total rows before handling negative generation: 23275
Total rows after removing negative generation anomalies: 22978
Anomalies isolated: 297 rows flagged.

All cleaned and standardized datasets successfully saved to data/processed/!
